In [ ]:
import time

In [1]:
# Initial imports

from datetime import datetime, timezone
import os
import errno
import re
import json
import logging
import pandas as pd
import teleGraph as tg

In [2]:
# Directory where you store scraped data and where you want to store processed data: 
homedir = 'telegraph_project'
utilspath = f'./{homedir}/'
datapath = f'./{homedir}/scraped_data/' #scraped data
cleanpath = f'./{homedir}/clean_data/' #cleaned data
respath = f'./{homedir}/data/' #cleaned data

for path in [utilspath, datapath, cleanpath, respath]:
    try:
        os.mkdir(path)
    except OSError as e:
        if e.errno != errno.EEXIST:
            raise

In [3]:
#logger config
logger = logging.getLogger(__name__)
logging.basicConfig(filename=utilspath + 'teleprep.log', filemode='w', level=logging.WARNING)

In [4]:
#  PARAMETERS:
#  Here you can select the time window you would like to extract data from the scraped files:
date_min = '2025-12-15 00:00:00' # @param {type:"date"}
date_max = '2026-04-12 23:59:59' # @param {type:"date"}
date_min = datetime.fromisoformat(date_min).replace(tzinfo=timezone.utc)
date_max = datetime.fromisoformat(date_max).replace(tzinfo=timezone.utc)
# Choose the format of the files you collect: `excel` or `parquet`:
File = 'excel' # @param ["excel", "parquet"]
if File == 'excel':
   file_extension = r'xlsx'
elif File == 'parquet':
    file_extension = r'parquet'
# Regex for valid filenames in directory:
data_files = fr'''
    (?<=complete\_)                     # Positive Lookbehind,
                                        # search for strings starting with a valid prefix:
                                        # 'complete_'
    (\@\w+\_\d*)                        # @username + user_id
    (?=\_msg\_data\.{file_extension})   # Positive Lookahead,
                                        # search only for files ending with a given suffix
                                        # '_msg_data.file_extension (xlsx or parquet)
    '''
data_files = re.compile(data_files, flags = re.VERBOSE | re.ASCII)
#language model
lang_model = tg.get_fasttext()

# Something something
filename = "telegraph_poligon" # @param {type:"string"}

In [5]:
#dictionary storing users in a format: username:{'entity_type':entity_type, 'entity_id':entity_id}
#used for caching users
peers_data_file = 'peers_data.json' # @param {type:"string"}
#check if premade version exists:
try:
    with open(utilspath + f'{peers_data_file}', 'r', encoding='utf-8') as f:
        peers_data = json.loads(f.read())
#if not, create a new one:
except Exception as e:
    peers_data = tg.extract_peers_from_dir(datapath, data_files)

In [6]:
#JSON file with channels classification:
channels_categories = 'channels_categories.json'
try:
    with open(utilspath + f'{channels_categories}', 'r', encoding='utf-8') as f:
        channels_categories = json.loads(f.read())
#if not provided, return simple user/channel/chat split:
except Exception as e:
    channels_categories = {0: "User", 1: "Channel", 2: "Chat"}

#JSON file with preliminarily classified channels
channels_classified = 'channels_classified.json'
try:
    with open(utilspath + f'{channels_classified}', 'r', encoding='utf-8') as f:
        channels_classified = json.loads(f.read())
#if not provided, return None:
except Exception as e:
    channels_classified = None

In [9]:
keywords = re.compile(r'(((NATO|) (bandym\w+|karin\w+)|Kap(č|c)iamies\w+) poligon\w+(\sne|))|(Kap(č|c)iamies\w+)|((полигон(\w+|) \w+ |)Капчяместис\w+)',
                      flags = re.UNICODE)
keywordsdf = pd.DataFrame()
keywords_colname = "Poligon Mentioned"

In [10]:
for fname in os.listdir(datapath):
    #extract name and id from valid files:
    matched = data_files.search(fname)
    if not matched:
        continue
    channel, channel_id = matched.group().rsplit('_', 1) #split string into channel name and channel id
    #open file:
    if File == 'excel':
       df = pd.read_excel(datapath + fname, engine='openpyxl')
    elif File == 'parquet':
        df = pd.read_parquet(datapath + fname, engine='pyarrow')
    if df.empty:
        print(f'{fname} file is empty!')
        continue 
    #update missing values:
    tg.update_missing_peers(df, peers_data)
    #clean text (remove links and emojis):
    df['Content'] = df['Content'].map(lambda x: tg.replace_links(str(x)))
    #if anonymize:
        #replace missing usernames:
     ##   df['Content'] = df['Content'].map(lambda x: tg.replace_usernames(x, peers_data, with_ids = True))
   # else:
    df['Content'] = df['Content'].map(lambda x: tg.replace_usernames(x, peers_data, with_ids = False))
    #language prediction:
    df['Language'] = df['Content'].map(lambda x: tg.predict_post_language(x, lang_model.predict).split('_', 4)[-1])
    #check for keywords mentions:
    keywords_mentions = df['Content'].map(lambda x: False if not keywords.findall(str(x)) else True)
    df[keywords_colname] = keywords_mentions
    keywordsdf = pd.concat([keywordsdf, df[ keywords_mentions]]).reset_index(drop=True)
    #save cleaned dfs:
    tg.save_data(df, cleanpath + fname.split('.', 1)[0], File)
#save keyword df:
tg.save_data(keywordsdf, respath + "keyword_df", File)

complete_@baltologija__msg_data.xlsx file is empty!
complete_@Jaunieji_Partizanai__msg_data.xlsx file is empty!
complete_@Lietuvar__msg_data.xlsx file is empty!
complete_@lithuanianews24__msg_data.xlsx file is empty!
complete_@LitvaBelarus__msg_data.xlsx file is empty!
complete_@pglnk_lt__msg_data.xlsx file is empty!
complete_@Pilietis_1__msg_data.xlsx file is empty!
complete_@tiesioginedemokratija_1175857193_msg_data.xlsx file is empty!


### Get full edgelist + metadata:

In [22]:
nodes_metadata = tg.create_peer_metadata()
edgelist_full = tg.get_edgelist(cleanpath, nodes_metadata = nodes_metadata, filenames = data_files, variant = 'all')
edgelist_full.to_csv(respath + f'edgelist_full.csv', index = False, header = True, mode='w', sep = ';')
#save edges metadata:
with open(respath + f'full_nodes_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(nodes_metadata, f, ensure_ascii=False, indent=4, default=list)

### Get @Kapciamiesciopoligonas edgelist + metadata:

In [23]:
kapcio_nodes_metadata = tg.create_peer_metadata()
kacpiodf = pd.read_excel(cleanpath + 'complete_@Kapciamiesciopoligonas_3137388250_msg_data.xlsx', engine='openpyxl')
edgelist_kapcio = tg.get_edgelist(kacpiodf, nodes_metadata = kapcio_nodes_metadata, 
                                  respath = respath + "kapcio_edgelist.csv", variant = 'all')
#save edges metadata:
with open(respath + f'kapcio_nodes_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(kapcio_nodes_metadata, f, ensure_ascii=False, indent=4, default=list)

In [24]:
channel_idx = 3137388250
kapcio_subgraph_edgelist = tg.get_peer_subgraph(channel_idx, edgelist_full, target_only = False, get_all = True)
kapcio_subgraph_edgelist.to_csv(respath + f'kapcio_subgraph_edgelist.csv', index = False, header = True, mode='w', sep = ';')

### Get poligon mentions edgelist + metadata:

In [25]:
keywords_nodes_metadata = tg.create_peer_metadata()
keywordsdf = pd.read_excel(respath + "keyword_df.xlsx", engine='openpyxl')
edgelist_keywords = tg.get_edgelist(keywordsdf, nodes_metadata = keywords_nodes_metadata, 
                                  respath = respath + "keywords_edgelist.csv", variant = 'all')
#save edges metadata:
with open(respath + f'keywords_nodes_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(keywords_nodes_metadata, f, ensure_ascii=False, indent=4, default=list)

In [26]:
keywords_iterator = keywordsdf[['Author ID', 'Message ID']].itertuples(index=False, name=None)
keywords_subgraph_edgelist = tg.get_posts_subgraph(keywords_iterator, edgelist_full, target_only = False, get_all = True)
keywords_subgraph_edgelist.to_csv(respath + f'keywords_subgraph_edgelist.csv', index = False, header = True, mode='w', sep = ';')